# RetNet 272M · 8K 上下文 · Kaggle T4×2 训练

三个交付物：本 notebook + 数据集 zip + 初始权重 zip。

流程：① 改第 1 个 cell 的路径配置 → ② 依次运行所有 cell。

精度自动选择：有 bf16 用 bf16，否则（T4 无 bf16 TensorCore）回退 fp32。


In [1]:
# ============================================================
# 【第 1 个 cell】配置区 —— 所有路径与超参数都在这里，改完直接运行
# ============================================================
import os, json

# ---- 路径配置（按你上传到 Kaggle 的 Dataset 名称修改）----
# Kaggle 会把上传的 zip 自动解压到 /kaggle/input/<dataset名>/
DATASET_DIR = "/kaggle/input/datasets/ayaka157/retnet-v11-8k-dataset/chat_pack_v11"      # 数据集
MODEL_DIR   = "/kaggle/input/datasets/ayaka157/retnet-272m-init-weights-2026-8-25/retnet_272m_rp"  # 初始权重
OUTPUT_DIR  = "/kaggle/working/retnet_272m_v11_8k"             # 输出(checkpoint + 最终权重)

# ---- 训练超参数 ----
SEQ_LEN          = 8192   # 上下文长度（数据行宽 = SEQ_LEN + 1）
PER_DEVICE_BATCH = 2      # 每张卡 batch；2 卡 DDP 总 batch = 4
GRAD_ACCUM       = 1      # 梯度累积；有效 batch = PER_DEVICE_BATCH * GPU数 * GRAD_ACCUM
MAX_STEPS        = 30000
LR               = 1.1e-4
MIN_LR           = 1.1e-5
WARMUP           = 300
CHUNK_SIZE       = 64
SAVE_STEPS       = 2500
EVAL_STEPS       = 2500
SAVE_LIMIT       = 3
WEIGHT_DECAY     = 0.1
MAX_GRAD_NORM    = 1.0
USE_COMPILE      = False  # torch.compile 提速（首次编译慢，T4 上慎开，失败自动回退 eager）

# ---- 显存参考（fp32 每卡）----
# 固定区(参数+梯度+Adam fp32) ~4.36GB + logits/CE 各 b*0.655GB + GC 保存输入 b*0.67GB + 反传重算 ~2GB
#   b=2 -> ~10.3GB（安全）  b=3 -> ~12.3GB  b=4 -> ~14.3GB（贴近 16GB，可能溢出）
# OOM 就把 PER_DEVICE_BATCH 往下调。

# ---- 路径合法性校验（不合法立刻抛错）----
def _must_dir(p, name):
    if not os.path.isdir(p):
        raise FileNotFoundError(
            f"[配置错误] {name} 目录不存在: {p}\n"
            f"请确认 Kaggle Dataset 名称与上面路径一致。")
def _must_file(p, name):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"[配置错误] {name} 文件不存在: {p}")

_must_dir(DATASET_DIR, "DATASET_DIR")
_must_dir(MODEL_DIR, "MODEL_DIR")
for f in ("train.ids.npy", "train.wcode.npy", "val.ids.npy", "val.wcode.npy"):
    _must_file(os.path.join(DATASET_DIR, f), f"数据集/{f}")
for f in ("config.json", "model.safetensors"):
    _must_file(os.path.join(MODEL_DIR, f), f"权重/{f}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CFG = dict(
    dataset_dir=DATASET_DIR, model_dir=MODEL_DIR, output_dir=OUTPUT_DIR,
    seq_len=SEQ_LEN, per_device_batch=PER_DEVICE_BATCH, grad_accum=GRAD_ACCUM,
    max_steps=MAX_STEPS, lr=LR, min_lr=MIN_LR, warmup=WARMUP, chunk_size=CHUNK_SIZE,
    save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS, save_limit=SAVE_LIMIT,
    weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM, use_compile=USE_COMPILE,
)
with open("/kaggle/working/config.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)

print("[OK] 路径校验通过，配置已写入 /kaggle/working/config.json")
print(f"  数据集 : {DATASET_DIR}")
print(f"  权重   : {MODEL_DIR}")
print(f"  输出   : {OUTPUT_DIR}")
print(f"  seq={SEQ_LEN}  per_device_batch={PER_DEVICE_BATCH}  grad_accum={GRAD_ACCUM}  max_steps={MAX_STEPS}")


[OK] 路径校验通过，配置已写入 /kaggle/working/config.json
  数据集 : /kaggle/input/datasets/ayaka157/retnet-v11-8k-dataset/chat_pack_v11
  权重   : /kaggle/input/datasets/ayaka157/retnet-272m-init-weights-2026-8-25/retnet_272m_rp
  输出   : /kaggle/working/retnet_272m_v11_8k
  seq=8192  per_device_batch=2  grad_accum=1  max_steps=30000


In [2]:
# ============================================================
# 【第 2 个 cell】安装依赖
# ============================================================
import subprocess, sys
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# torch 在 Kaggle 已预装；补装/对齐训练栈
_pip("transformers<5", "accelerate", "safetensors")

import torch
print("torch        :", torch.__version__, "| cuda:", torch.version.cuda)
print("transformers :", __import__("transformers").__version__)
print("accelerate   :", __import__("accelerate").__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 554.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
torch        : 2.10.0+cu128 | cuda: 12.8
transformers : 4.57.6
accelerate   : 1.13.0


In [3]:
# ============================================================
# 【第 3 个 cell】检测 GPU 与精度（T4 无 bf16 -> 回退 fp32）
# ============================================================
import torch

N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 2, f"需要 2 张 GPU，当前只有 {N_GPUS} 张"
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU{i}: {p.name}  {p.total_memory / 2**30:.1f} GiB  (cap {torch.cuda.get_device_capability(i)})")

cap = torch.cuda.get_device_capability(0)
PRECISION = "bf16" if cap[0] >= 8 else "fp32"
print(f"precision = {PRECISION}" + ("  (Ampere+ 原生 bf16)" if PRECISION == "bf16"
                                    else "  (Turing 无 bf16 TensorCore，按约定回退 fp32)"))


GPU0: Tesla T4  14.6 GiB  (cap (7, 5))
GPU1: Tesla T4  14.6 GiB  (cap (7, 5))
precision = fp32  (Turing 无 bf16 TensorCore，按约定回退 fp32)


In [6]:
%%writefile /kaggle/working/train_retnet.py
"""RetNet 272M 8K DDP 训练脚本（由 notebook 生成）。"""
from __future__ import annotations

import json
import os
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (PreTrainedModel, PretrainedConfig, TrainingArguments,
                          Trainer)

# ==================== 模型定义（与 checkpoint 状态字典键逐字一致）====================
class RetNetHFConfig(PretrainedConfig):
    model_type = "retnet_hf"

    def __init__(self, vocab_size=8192, d_model=512, num_layers=6, num_heads=8,
                 value_dim=None, ffn_dim=None, dropout=0.1, norm_eps=1e-5,
                 tie_weights=True, embed_scale=None, prompt_loss_weight=0.1,
                 use_gla=False, use_swiglu=False, use_growth_ln=False,
                 growth_old_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.value_dim = value_dim if value_dim is not None else 2 * d_model
        self.ffn_dim = ffn_dim if ffn_dim is not None else 2 * d_model
        self.dropout = dropout
        self.norm_eps = norm_eps
        self.tie_weights = tie_weights
        self.embed_scale = embed_scale
        self.prompt_loss_weight = prompt_loss_weight
        self.use_gla = use_gla
        self.use_swiglu = use_swiglu
        self.use_growth_ln = use_growth_ln
        self.growth_old_dim = growth_old_dim

    @property
    def key_dim(self) -> int:
        return self.d_model // self.num_heads


def rotate_every_two(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    x_rot = torch.stack((-x2, x1), dim=-1)
    return x_rot.flatten(-2)


def theta_shift(x: torch.Tensor, sin: torch.Tensor, cos: torch.Tensor) -> torch.Tensor:
    return x * cos + rotate_every_two(x) * sin


class MultiScaleRetention(nn.Module):
    def __init__(self, config: RetNetHFConfig):
        super().__init__()
        d_model, num_heads, value_dim = config.d_model, config.num_heads, config.value_dim
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        if value_dim % num_heads != 0:
            raise ValueError("value_dim must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.key_dim = d_model // num_heads
        self.value_dim = value_dim
        self.value_head_dim = value_dim // num_heads
        self.scale = self.key_dim ** -0.5

        gamma = 1.0 - torch.pow(2.0, -5.0 - torch.arange(num_heads, dtype=torch.float32))
        if config.use_gla:
            # GLA: 可学习逐头门控（初始值 = RetNet 固定公式，训练中自适应）
            self.gamma = nn.Parameter(gamma.clone())
        else:
            self.register_buffer("gamma", gamma, persistent=True)
        inv_freq = 1.0 / (10000.0 ** torch.linspace(0.0, 1.0, steps=self.key_dim // 2,
                                                     dtype=torch.float32))
        self.register_buffer("inv_freq", inv_freq.repeat_interleave(2), persistent=False)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, value_dim, bias=False)
        self.g_proj = nn.Linear(d_model, value_dim, bias=False)
        self.out_proj = nn.Linear(value_dim, d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)
        self.group_norm = nn.GroupNorm(num_heads, value_dim, eps=config.norm_eps)

    def _init_weights(self):
        for name, p in self.named_parameters():
            if name.endswith("_proj.weight"):
                if name.split(".")[-1].startswith(("q_", "k_", "v_", "g_")):
                    nn.init.xavier_uniform_(p, gain=2 ** -2.5)
                else:
                    nn.init.xavier_uniform_(p, gain=1.0)
            elif isinstance(p, nn.Parameter) and p.dim() == 1:
                nn.init.zeros_(p)

    def _rotations(self, length, offset, device, dtype):
        position = torch.arange(offset, offset + length, device=device, dtype=torch.float32)
        freqs = torch.outer(position, self.inv_freq.to(device=device, dtype=torch.float32))
        return torch.sin(freqs).to(dtype=dtype), torch.cos(freqs).to(dtype=dtype)

    def _project_and_rotate(self, x, offset):
        bsz, tgt_len, _ = x.shape
        q = self.q_proj(x).view(bsz, tgt_len, self.num_heads, self.key_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, tgt_len, self.num_heads, self.key_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, tgt_len, self.num_heads, self.value_head_dim).transpose(1, 2)
        g = self.g_proj(x)
        k = k * self.scale
        sin, cos = self._rotations(tgt_len, offset, x.device, q.dtype)
        q = theta_shift(q, sin, cos)
        k = theta_shift(k, sin, cos)
        return q, k, v, g

    @staticmethod
    def _head_group_norm(x, num_heads, group_norm, bsz, tgt_len):
        x = x.transpose(1, 2).reshape(bsz * tgt_len, num_heads * x.shape[-1])
        x = group_norm(x.unsqueeze(-1)).squeeze(-1)
        return x.reshape(bsz, tgt_len, -1)

    def _decay_mask(self, length, device, dtype):
        idx = torch.arange(length, device=device)
        delta = idx[:, None] - idx[None, :]
        causal = delta >= 0
        exp = delta.masked_fill(~causal, 0)
        mask = self.gamma.to(device=device, dtype=torch.float32)[:, None, None] ** exp
        mask = mask.masked_fill(~causal, 0.0)
        mask = mask / mask.sum(dim=-1, keepdim=True).sqrt().clamp_min(1e-8)
        return mask.to(dtype=dtype)

    def _parallel_retention(self, q, k, v):
        bsz, _, tgt_len, _ = q.shape
        mask = self._decay_mask(tgt_len, q.device, q.dtype)
        scores = torch.matmul(q, k.transpose(-1, -2)) * mask
        scores = scores / scores.detach().abs().sum(dim=-1, keepdim=True).clamp_min(1.0)
        return torch.matmul(scores, v)

    def _parallel_final_state(self, k, v, state):
        bsz, _, tgt_len, key_dim = k.shape
        idx = torch.arange(tgt_len, device=k.device)
        weights = self.gamma.to(device=k.device, dtype=torch.float32)[:, None] ** (tgt_len - 1 - idx)
        new_state = torch.einsum("bhtk,bhtv,ht->bhkv", k, v, weights.to(k.dtype))
        if state is not None:
            gamma_pow = self.gamma.to(device=k.device, dtype=k.dtype) ** tgt_len
            new_state = gamma_pow.view(1, -1, 1, 1) * state + new_state
        return new_state

    def _recurrent_retention(self, q, k, v, state):
        bsz, _, tgt_len, key_dim = q.shape
        value_head_dim = v.shape[-1]
        if state is None:
            state = torch.zeros(bsz, self.num_heads, key_dim, value_head_dim,
                                device=q.device, dtype=q.dtype)
        outputs = []
        gamma = self.gamma.to(device=q.device, dtype=q.dtype)
        for t in range(tgt_len):
            state = gamma.view(1, -1, 1, 1) * state + \
                k[:, :, t, :].unsqueeze(-1) * v[:, :, t, :].unsqueeze(-2)
            out_t = torch.sum(q[:, :, t, :].unsqueeze(-1) * state, dim=-2)
            outputs.append(out_t)
        output = torch.stack(outputs, dim=2)
        return output, state

    def _chunkwise_retention(self, q, k, v, state, chunk_size):
        """Equation (7): inner-chunk parallel + cross-chunk recurrent state."""
        bsz, _, tgt_len, key_dim = q.shape
        if tgt_len % chunk_size != 0:
            raise ValueError(f"chunkwise mode requires seq_len ({tgt_len}) to be a "
                             f"multiple of chunk_size ({chunk_size})")
        if state is None:
            state = torch.zeros(bsz, self.num_heads, key_dim, v.shape[-1],
                                device=q.device, dtype=q.dtype)
        gamma = self.gamma.to(device=q.device, dtype=q.dtype)
        outputs = []
        rel = torch.arange(chunk_size, device=q.device)
        delta = rel[:, None] - rel[None, :]
        causal = delta >= 0
        inner_mask = torch.where(causal, gamma[:, None, None] ** delta.masked_fill(~causal, 0),
                                 torch.zeros_like(delta, dtype=torch.float32))
        cross_decay = gamma[:, None] ** (rel + 1)[None, :]
        value_decay = gamma[:, None] ** (chunk_size - 1 - rel)[None, :]
        chunk_decay = gamma ** chunk_size

        for i in range(tgt_len // chunk_size):
            sl = slice(i * chunk_size, (i + 1) * chunk_size)
            qi, ki, vi = q[:, :, sl], k[:, :, sl], v[:, :, sl]
            inner_scores = torch.matmul(qi, ki.transpose(-1, -2)) * inner_mask
            inner = torch.matmul(inner_scores, vi)
            cross = torch.matmul(qi * cross_decay[None, :, :, None], state)
            outputs.append(inner + cross)
            weighted_v = vi * value_decay[None, :, :, None]
            state = chunk_decay.view(1, -1, 1, 1) * state + \
                torch.matmul(ki.transpose(-1, -2), weighted_v)

        output = torch.cat(outputs, dim=2)
        return output, state

    def forward(self, x, mode="parallel", state=None, offset=0, return_state=False,
                chunk_size=64):
        bsz, tgt_len, _ = x.shape
        q, k, v, g = self._project_and_rotate(x, offset)

        if mode == "parallel":
            output, new_state = self._parallel_retention(q, k, v), None
            if return_state:
                new_state = self._parallel_final_state(k, v, state)
        elif mode == "recurrent":
            output, new_state = self._recurrent_retention(q, k, v, state)
        elif mode == "chunkwise":
            output, new_state = self._chunkwise_retention(q, k, v, state, chunk_size)
        else:
            raise ValueError(f"unknown mode: {mode}")

        output = self._head_group_norm(output, self.num_heads, self.group_norm, bsz, tgt_len)
        output = F.silu(g) * output
        output = self.dropout(output)
        output = self.out_proj(output)
        if return_state:
            return output, new_state
        return output


class GrowthLayerNorm(nn.Module):
    """Two-segment LayerNorm for function-preserving width growth.

    The old ``old_dim`` channels are normalized exactly as before (statistics
    unaffected by new channels); the new channels are scaled independently with
    ``gamma_new``/``beta_new`` and stay a zero side branch while
    ``gamma_new=0, beta_new=0``.  This is what makes pre-LN width expansion
    strictly function-preserving (BERT2BERT-style FPI for LayerNorm).

    state_dict keys: ``ln_old.weight``, ``ln_old.bias``, ``gamma_new``, ``beta_new``.
    """

    def __init__(self, old_dim: int, new_dim: int, eps: float = 1e-5,
                 gamma_new: float = 1.0, beta_new: float = 0.0):
        super().__init__()
        self.ln_old = nn.LayerNorm(old_dim, eps=eps)
        self.gamma_new = nn.Parameter(torch.full((new_dim - old_dim,), gamma_new))
        self.beta_new = nn.Parameter(torch.full((new_dim - old_dim,), beta_new))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        old = self.ln_old(x[..., : self.ln_old.normalized_shape[0]])
        new = x[..., self.ln_old.normalized_shape[0]:] * self.gamma_new + self.beta_new
        return torch.cat([old, new], dim=-1)


class RetNetBlock(nn.Module):
    def _norm(self, config: RetNetHFConfig, dim: int):
        if config.use_growth_ln:
            return GrowthLayerNorm(config.growth_old_dim, dim, eps=config.norm_eps,
                                   gamma_new=1.0, beta_new=0.0)
        return nn.LayerNorm(dim, eps=config.norm_eps)

    def __init__(self, config: RetNetHFConfig):
        super().__init__()
        self.msr = MultiScaleRetention(config)
        self.norm1 = self._norm(config, config.d_model)
        self.gradient_checkpointing = False
        if config.use_swiglu:
            # SwiGLU FFN：y = silu(xW_gate) * (xW_up) -> W_down
            self.ffn = nn.Sequential()  # placeholder; real modules below
            self.gate_proj = nn.Linear(config.d_model, config.ffn_dim, bias=False)
            self.up_proj = nn.Linear(config.d_model, config.ffn_dim, bias=False)
            self.down_proj = nn.Linear(config.ffn_dim, config.d_model, bias=False)
            self.swiglu = True
        else:
            self.ffn = nn.Sequential(
                nn.Linear(config.d_model, config.ffn_dim, bias=False),
                nn.GELU(),
                nn.Linear(config.ffn_dim, config.d_model, bias=False),
            )
            self.swiglu = False
        self.norm2 = self._norm(config, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def _block_forward(self, x, mode, state, offset, return_state, chunk_size=64):
        msr_out = self.msr(self.norm1(x), mode=mode, state=state, offset=offset,
                           return_state=return_state, chunk_size=chunk_size)
        if return_state:
            y, new_state = msr_out
        else:
            y, new_state = msr_out, None
        y = x + y
        if self.swiglu:
            ffn_out = F.silu(self.gate_proj(self.norm2(y))) * self.up_proj(self.norm2(y))
            ffn_out = self.down_proj(ffn_out)
        else:
            ffn_out = self.ffn(self.norm2(y))
        y = y + self.dropout(ffn_out)
        if return_state:
            return y, new_state
        return y

    def forward(self, x, mode="parallel", state=None, offset=0, return_state=False,
                chunk_size=64):
        if self.gradient_checkpointing and self.training:
            return torch.utils.checkpoint.checkpoint(
                self._block_forward, x, mode, state, offset, return_state, chunk_size,
                use_reentrant=False)
        return self._block_forward(x, mode, state, offset, return_state, chunk_size)


class RetNetForCausalLM(PreTrainedModel):
    """RetNet language model with a from-scratch token embedding.

    Forward signature is Trainer-compatible:
        forward(input_ids, labels=None, wcodes=None, ...) -> CausalLMOutput
    When labels+wcodes are given the loss is the wcode-weighted cross entropy
    (wcode: 0=pad, 1=reply, 2=prompt, 3=plain LM) exactly like train.py.
    """
    config_class = RetNetHFConfig
    base_model_prefix = "retnet"
    supports_gradient_checkpointing = True

    def __init__(self, config: RetNetHFConfig):
        super().__init__(config)
        self.embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embed_dropout = nn.Dropout(config.dropout)
        self.embed_scale = config.embed_scale if config.embed_scale is not None \
            else config.d_model ** 0.5
        self.blocks = nn.ModuleList([RetNetBlock(config) for _ in range(config.num_layers)])
        if config.use_growth_ln:
            self.final_norm = GrowthLayerNorm(config.growth_old_dim, config.d_model,
                                              eps=config.norm_eps,
                                              gamma_new=1.0, beta_new=0.0)
        else:
            self.final_norm = nn.LayerNorm(config.d_model, eps=config.norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        if config.tie_weights:
            self.lm_head.weight = self.embedding.weight
            self._tied_weights_keys = ["lm_head.weight"]
        self._init_weights_local()

    def _init_weights_local(self):
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        for block in self.blocks:
            if block.swiglu:
                nn.init.xavier_uniform_(block.gate_proj.weight)
                nn.init.xavier_uniform_(block.up_proj.weight)
                nn.init.xavier_uniform_(block.down_proj.weight)
            else:
                for name, p in block.ffn.named_parameters():
                    if name.endswith("weight"):
                        nn.init.xavier_uniform_(p)
            block.msr._init_weights()
        if not self.config.tie_weights:
            nn.init.xavier_uniform_(self.lm_head.weight)

    def tie_weights(self):
        """共享 lm_head 与 embedding（from_pretrained 后恢复 tie）。"""
        if self.config.tie_weights:
            self.lm_head.weight = self.embedding.weight

    def _forward_hidden(self, input_ids, mode="parallel", states=None, offset=0,
                        return_state=False, chunk_size=64):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must be [batch, seq]")
        x = self.embedding(input_ids) * self.embed_scale
        x = self.embed_dropout(x)
        if states is None:
            states = [None] * len(self.blocks)
        if len(states) != len(self.blocks):
            raise ValueError("states must contain one entry per block")
        new_states: List[Optional[torch.Tensor]] = []
        for block, block_state in zip(self.blocks, states):
            if return_state or (mode == "recurrent" and block_state is not None):
                x, ns = block(x, mode=mode, state=block_state, offset=offset,
                              return_state=True, chunk_size=chunk_size)
                new_states.append(ns)
            else:
                x = block(x, mode=mode, state=block_state, offset=offset,
                          chunk_size=chunk_size)
                new_states.append(block_state)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        if return_state or mode == "recurrent":
            return logits, new_states
        return logits

    def forward(self, input_ids, labels=None, wcodes=None, mode=None,
                states=None, offset=0, return_state=False, chunk_size=64, **_kwargs):
        if mode is None:
            mode = getattr(self.config, "mode", "parallel")
        if return_state or mode == "recurrent" or mode == "chunkwise":
            logits, new_states = self._forward_hidden(input_ids, mode=mode, states=states,
                                                      offset=offset, return_state=True,
                                                      chunk_size=chunk_size)
        else:
            logits = self._forward_hidden(input_ids, mode=mode, states=states, offset=offset)
            new_states = None

        loss = None
        if labels is not None and wcodes is not None:
            logits_f = logits.float()
            token_loss = F.cross_entropy(
                logits_f.reshape(-1, logits_f.shape[-1]), labels.reshape(-1), reduction="none")
            w = torch.zeros_like(wcodes, dtype=torch.float32)
            w = torch.where(wcodes == 1, torch.ones_like(w), w)
            w = torch.where(wcodes == 2, torch.full_like(w, self.config.prompt_loss_weight), w)
            w = torch.where(wcodes == 3, torch.ones_like(w), w)
            loss = (token_loss * w.reshape(-1)).sum() / w.sum().clamp_min(1.0)
        elif labels is not None:
            logits_f = logits.float()
            loss = F.cross_entropy(logits_f.reshape(-1, logits_f.shape[-1]), labels.reshape(-1))

        from transformers.modeling_outputs import CausalLMOutput
        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 32,
                 temperature: float = 1.0, top_k: Optional[int] = None,
                 top_p: Optional[float] = None, repetition_penalty: float = 1.0,
                 eos_id: Optional[int] = None, device: Optional[torch.device] = None,
                 penalize_prompt: bool = False, no_repeat_ngram_size: int = 0) -> torch.Tensor:
        """Autoregressive generation (same algorithm/sampling as handwritten)."""
        device = device or next(self.parameters()).device
        ids = input_ids.to(device)
        states = [None] * len(self.blocks)
        if ids.shape[1] > 1:
            logits, states = self._forward_hidden(ids, mode="parallel", return_state=True)
            next_logits = logits[:, -1]
        else:
            next_logits, states = self._forward_hidden(ids, mode="recurrent", states=states,
                                                       offset=0, return_state=True)
            next_logits = next_logits[:, -1]
        generated = list(ids[0].tolist())
        prompt_len = ids.shape[1]
        offset = ids.shape[1]

        for _ in range(max_new_tokens):
            penalty_source = generated if penalize_prompt else generated[prompt_len:]
            banned = []
            if no_repeat_ngram_size and len(generated) >= no_repeat_ngram_size:
                n = no_repeat_ngram_size
                suffix = tuple(generated[-(n - 1):]) if n > 1 else ()
                for i in range(len(generated) - n + 1):
                    if tuple(generated[i:i + n - 1]) == suffix:
                        banned.append(generated[i + n - 1])
            if temperature > 0:
                logits = next_logits / temperature
                if repetition_penalty and repetition_penalty != 1.0:
                    for token_id in set(penalty_source):
                        score = logits[0, token_id]
                        logits[0, token_id] = score / repetition_penalty if score >= 0 \
                            else score * repetition_penalty
                for token_id in banned:
                    logits[0, token_id] = -torch.inf
                if top_k is not None:
                    k = min(top_k, logits.shape[-1])
                    top_values, _ = torch.topk(logits, k)
                    logits[logits < top_values[:, -1:]] = -torch.inf
                probs = F.softmax(logits, dim=-1)
                if top_p is not None and top_p < 1.0:
                    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
                    cumsum = sorted_probs.cumsum(dim=-1)
                    remove = cumsum > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    filtered = torch.zeros_like(probs)
                    filtered.scatter_(1, sorted_indices[~remove].unsqueeze(0),
                                      sorted_probs[~remove].unsqueeze(0))
                    probs = filtered / filtered.sum(dim=-1, keepdim=True)
                next_id = torch.multinomial(probs, num_samples=1).item()
            else:
                logits = next_logits.clone()
                for token_id in banned:
                    logits[0, token_id] = -torch.inf
                next_id = logits.argmax(dim=-1).item()
            if eos_id is not None and next_id == eos_id:
                break
            generated.append(next_id)
            next_token = torch.tensor([[next_id]], device=device, dtype=torch.long)
            next_logits, states = self._forward_hidden(next_token, mode="recurrent",
                                                       states=states, offset=offset,
                                                       return_state=True)
            next_logits = next_logits[:, -1]
            offset += 1
        return torch.tensor([generated], device=device, dtype=torch.long)


# 补丁：确保梯度检查点在各 transformers 版本下都可靠开启
def _set_gradient_checkpointing(self, module, value=False, **kwargs):
    if isinstance(module, RetNetBlock):
        module.gradient_checkpointing = value
if not hasattr(RetNetForCausalLM, "_set_gradient_checkpointing"):
    RetNetForCausalLM._set_gradient_checkpointing = _set_gradient_checkpointing
RetNetForCausalLM._supports_gradient_checkpointing = True


# ==================== 数据 ====================
class PackedChatDataset(Dataset):
    """行布局 (seq_len+1)：input_ids=row[:-1], labels=row[1:], wcodes=codes[1:]"""
    def __init__(self, ids_path, wcode_path, seq_len):
        self.ids = np.load(ids_path, mmap_mode="r")
        self.codes = np.load(wcode_path, mmap_mode="r")
        assert self.ids.ndim == 2 and self.ids.shape == self.codes.shape
        assert self.ids.shape[1] == seq_len + 1, \
            f"rows={self.ids.shape[1]} != seq_len+1={seq_len + 1}"
        self.seq_len = seq_len

    def __len__(self):
        return self.ids.shape[0]

    def __getitem__(self, idx):
        row = self.ids[idx]
        return {
            "input_ids": torch.from_numpy(row[:-1].astype(np.int64)),
            "labels": torch.from_numpy(row[1:].astype(np.int64)),
            "wcodes": torch.from_numpy(self.codes[idx][1:].astype(np.int64)),
        }


class ChatDataCollator:
    def __call__(self, batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch]),
            "wcodes": torch.stack([b["wcodes"] for b in batch]),
        }


# ==================== 主流程 ====================
def main():
    cfg = json.load(open("/kaggle/working/config.json", encoding="utf-8"))

    cap = torch.cuda.get_device_capability(0)
    precision = "bf16" if cap[0] >= 8 else "fp32"

    model = RetNetForCausalLM.from_pretrained(cfg["model_dir"])
    model.config.mode = "chunkwise"
    model.config.chunk_size = cfg["chunk_size"]
    # 显式打开梯度检查点（不依赖 Trainer 内部行为）
    model.gradient_checkpointing_enable()
    for b in model.blocks:
        b.gradient_checkpointing = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    ds = cfg["dataset_dir"]
    train_ds = PackedChatDataset(os.path.join(ds, "train.ids.npy"),
                                 os.path.join(ds, "train.wcode.npy"), cfg["seq_len"])
    val_ds = PackedChatDataset(os.path.join(ds, "val.ids.npy"),
                               os.path.join(ds, "val.wcode.npy"), cfg["seq_len"])

    training_args = TrainingArguments(
        output_dir=cfg["output_dir"],
        overwrite_output_dir=True,
        max_steps=cfg["max_steps"],
        per_device_train_batch_size=cfg["per_device_batch"],
        per_device_eval_batch_size=cfg["per_device_batch"],
        gradient_accumulation_steps=cfg["grad_accum"],
        learning_rate=cfg["lr"],
        lr_scheduler_type="cosine_with_min_lr",
        lr_scheduler_kwargs={"min_lr": cfg["min_lr"]},
        warmup_steps=cfg["warmup"],
        weight_decay=cfg["weight_decay"],
        max_grad_norm=cfg["max_grad_norm"],
        bf16=(precision == "bf16"),
        fp16=False,                      # 约定：bf16 或 fp32，绝不用 fp16（8K 会 NaN）
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=cfg["eval_steps"],
        save_strategy="steps",
        save_steps=cfg["save_steps"],
        save_total_limit=cfg["save_limit"],
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to=[],
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        seed=1234,
        gradient_checkpointing=True,
        ddp_find_unused_parameters=True,  # tied weights 安全
        include_inputs_for_metrics=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=ChatDataCollator(),
    )

    if cfg["use_compile"]:
        try:
            trainer._set_signature_columns_if_needed()
            trainer.model = torch.compile(trainer.model, backend="inductor")
            print("[info] torch.compile(inductor) applied", flush=True)
        except Exception as e:
            print(f"[warn] torch.compile failed ({e}); continuing eager", flush=True)

    print(f"[info] precision={precision}  rank={os.environ.get('LOCAL_RANK')}  "
          f"train_rows={len(train_ds)}  val_rows={len(val_ds)}", flush=True)

    trainer.train()

    if trainer.is_world_process_zero():
        trainer.save_model(cfg["output_dir"])
        print("[done] final weights saved to", cfg["output_dir"], flush=True)


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_retnet.py


In [7]:
# ============================================================
# 【第 5 个 cell】启动 DDP 训练（2 卡 torch.distributed.run）
# ============================================================
import subprocess, sys

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--nproc_per_node", str(N_GPUS),
    "--master_port", "29500",
    "/kaggle/working/train_retnet.py",
]
print("launching:", " ".join(cmd), flush=True)
subprocess.check_call(cmd)
print("[done] training finished; weights in /kaggle/working/retnet_272m_v11_8k/")


launching: /usr/bin/python3 -m torch.distributed.run --nproc_per_node 2 --master_port 29500 /kaggle/working/train_retnet.py



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Traceback (most recent call last):
Traceback (most recent call last):
  File "/kaggle/working/train_retnet.py", line 609, in <module>
  File "/kaggle/working/train_retnet.py", line 609, in <module>
        main()
main()
  File "/kaggle/working/train_retnet.py", line 528, in main
  File "/kaggle/working/train_retnet.py", line 528, in main
        cfg = json.load(open("/kaggle/working/config.json", encoding="utf-8"))
cfg = json.load(open("/kaggle/working/config.json", encoding="utf-8"))
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^
FileNotFoundError: FileNotFoundError[Errno 2] No such file or direc

CalledProcessError: Command '['/usr/bin/python3', '-m', 'torch.distributed.run', '--nproc_per_node', '2', '--master_port', '29500', '/kaggle/working/train_retnet.py']' returned non-zero exit status 1.